In [16]:
import numpy as np
import pandas as pd
import os
from os.path import join, exists

from PIL import Image
from skimage.transform import resize
from tqdm import tqdm
import copy
import albumentations as A

import matplotlib.pyplot as plt
from math import ceil

In [17]:
BASE_PATH = "/home/orrin/aps360/pediatric-cxr-model/"
DATA_BASE_PATH = join(BASE_PATH, "data_cleaned/")
SAVE_PATH_IMAGES = join(DATA_BASE_PATH, "images/")
SAVE_PATH_ANNOTATIONS = join(DATA_BASE_PATH, "annotations/")

In [18]:
# define standard pandas df column names
column_names = [
    "filename",
    "no finding",
    "bronchitis",
    "broncho-pnuemonia",
    "pneumonia",
    "bronchiolitis",
    "heart disease",
    "atelectasis",
    "consolidation",
    "infiltration",
    "pneumothorax",
    "effusion",
    "other",
    "has bbox",
    "min_x",
    "min_y",
    "width",
    "height",
]

# helper function to add to df
def add_to_df(df, filename, label, min_x=None, min_y=None, width=None, height=None):
    # make sure label in valid label names
    if label not in column_names:
        raise ValueError(f"Label {label} not in column names")
    # make sure either all bbox values are None or all bbox values are not None
    if min_x is not None and (min_y is None or width is None or height is None) or \
        min_y is not None and (min_x is None or width is None or height is None) or \
        width is not None and (min_x is None or min_y is None or height is None) or \
        height is not None and (min_x is None or min_y is None or width is None):
        raise ValueError("Either all bbox values must be None or all bbox values must be not None")
        
    row = {
        "filename": filename,
        "label" : label,
        "has bbox": 1 if min_x is not None else 0,
        "min_x": min_x,
        "min_y": min_y,
        "width": width,
        "height": height,
    }
    df.loc[len(df)] = row

# helper function to visualize a single image and its bounding box
def visualize_image_with_bbox(df, row_index):
    # get row and image
    row = df.iloc[row_index]
    img_path = join(SAVE_PATH_IMAGES, row["filename"])
    img = np.array(Image.open(img_path).convert("L"))
    # get label from one-hot encoding row
    label = None
    for col in column_names[1:13]:  # skip filename and bbox columns
        if row[col] == 1:
            label = col
            break
    # plot
    plt.imshow(img, cmap="gray")
    if row["has bbox"] == 1:
        x_min = row["min_x"]
        y_min = row["min_y"]
        width = row["width"]
        height = row["height"]
        rect = plt.Rectangle((x_min, y_min), width, height, edgecolor="red", facecolor="none")
        plt.gca().add_patch(rect)
    plt.title(f"{row["filename"]} - {label}")
    plt.axis("off")
    plt.show()

In [19]:
# Read all labels from master csv
df = pd.read_csv(join(SAVE_PATH_ANNOTATIONS, "master_annotations_alternate.csv"))
value_counts = df['label'].value_counts()
n = len(df)
perc_value_counts = value_counts / n
print(value_counts)
print('----------------')
print(perc_value_counts)

label
no finding           11382
pneumonia             4722
other                 1477
bronchitis            1016
heart disease          765
infiltration           756
broncho-pnuemonia      577
bronchiolitis          519
effusion               306
consolidation          227
Name: count, dtype: int64
----------------
label
no finding           0.523383
pneumonia            0.217133
other                0.067917
bronchitis           0.046719
heart disease        0.035177
infiltration         0.034763
broncho-pnuemonia    0.026532
bronchiolitis        0.023865
effusion             0.014071
consolidation        0.010438
Name: count, dtype: float64


In [20]:
df_new = copy.deepcopy(df)
labels = value_counts.keys()
perc_wanted = 0.05
tot_augment = 0
augmented_by_label = {}

# define aumentation sequence
transform = A.Compose([
    A.Affine(
        scale=(0.8, 1.2),
        rotate=(-20,20),
        translate_percent=(-0.1, 0.1),
        p=1.0
    ),
    # A.ThinPlateSpline(scale=(0.01, .05), p=0.2.0),
    A.RandomBrightnessContrast(
        brightness_limit=0.2,
        contrast_limit=0.2,
        p=0.3
        ),
    A.RandomToneCurve(p=0.1),
    A.Normalize(normalization='min_max', p=1.0)
], bbox_params=A.BboxParams(coord_format='coco'))

for lab in labels:
    # find number of augmented images needed to get data count up to 5%
    count = value_counts[lab]
    aug_num = max(0, ceil(perc_wanted * n - count))
    print(f"Creating {aug_num} augmented images for {lab}")
    if aug_num == 0:
        continue

    mask = df['label'] == lab
    indices = np.where(mask)[0]
    n_indices = len(indices)
    for i in tqdm(range(aug_num)):
        # randomly sample image to augment
        idx = np.random.randint(0, n_indices)
        row = df.iloc[indices[idx]]
        fname = row['filename']
        img_path = join(SAVE_PATH_IMAGES, fname)

        img = np.array(Image.open(img_path).convert('L'))
        if row['has bbox']:
            bbox = [row['min_x'], row['min_y'], row['width'], row['height']]
            augmented = transform(image=img, bbox=bbox)
            image = augmented['image']
            bbox = augmented['bbox']
        else:
            augmented = transform(image=img)
            image = augmented['image']
        image = (image * 255).astype(np.uint8)  # convert back to uint8

        # # Plot for debugging
        # plt.imshow(image, cmap="gray")
        # if row["has bbox"] == 1:
        #     x_min = bbox[0]
        #     y_min = bbox[1]
        #     width = bbox[2]
        #     height = bbox[3]
        #     rect = plt.Rectangle((x_min, y_min), width, height, edgecolor="red", facecolor="none")
        #     plt.gca().add_patch(rect)
        # plt.title(f"{row["filename"]} - {lab}")
        # plt.axis("off")
        # plt.show()

        # create augmented row
        new_fname = f"augmented_{lab}_{i:03d}.jpg"
        if row['has bbox']:
            new_min_x = bbox[0]
            new_min_y = bbox[1]
            new_width = bbox[2]
            new_height = bbox[3]           
            add_to_df(
                df_new,
                filename=new_fname,
                label=lab,
                min_x=new_min_x,
                min_y=new_min_y,
                width=new_width,
                height=new_height
            )
        else:
            add_to_df(
                df_new,
                filename=new_fname,
                label=lab
            )

        # save image
        save_path = join(SAVE_PATH_IMAGES, new_fname)
        Image.fromarray(image).save(save_path)

        tot_augment += 1

# remove no finding (highest percentage) to keep same number of data
mask = df['label'] == 'no finding'
indices = np.where(mask)[0]
df_new.drop(indices[:tot_augment], inplace=True)

# save
df_new.to_csv(join(SAVE_PATH_ANNOTATIONS, "master_alternate_augmented.csv"), index=False)

Creating 0 augmented images for no finding
Creating 0 augmented images for pneumonia
Creating 0 augmented images for other
Creating 72 augmented images for bronchitis


  0%|          | 0/72 [00:00<?, ?it/s]

100%|██████████| 72/72 [00:00<00:00, 98.77it/s] 


Creating 323 augmented images for heart disease


100%|██████████| 323/323 [00:03<00:00, 81.58it/s]


...

100%|██████████| 569/569 [00:05<00:00, 95.94it/s] 


Creating 782 augmented images for effusion


100%|██████████| 782/782 [00:07<00:00, 100.18it/s]


Creating 861 augmented images for consolidation


100%|██████████| 861/861 [00:08<00:00, 97.62it/s] 


In [21]:
# new spread
value_counts = df_new['label'].value_counts()
n = len(df_new)
perc_value_counts = value_counts / n
print(value_counts)
print('----------------')
print(perc_value_counts)

label
no finding           7932
pneumonia            4722
other                1477
heart disease        1088
consolidation        1088
infiltration         1088
effusion             1088
bronchiolitis        1088
bronchitis           1088
broncho-pnuemonia    1088
Name: count, dtype: int64
----------------
label
no finding           0.364740
pneumonia            0.217133
other                0.067917
heart disease        0.050030
consolidation        0.050030
infiltration         0.050030
effusion             0.050030
bronchiolitis        0.050030
bronchitis           0.050030
broncho-pnuemonia    0.050030
Name: count, dtype: float64
